In [36]:
import sys
!{sys.executable} -m pip install pandas

  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 5.4 MB/s eta 0:00:0000:0100:01


# Theodorus Aaron Ugraha - 5027241056: spark/analysis.py
## Topik 8: HargaPangan - Monitor Harga Komoditas Bahan Pokok
3 Analisis Wajib:
  1. Volatilitas harga per komoditas
  2. Rata-rata harga per komoditas per periode
  3. Sebutan komoditas di berita (korelasi berita & harga)
Spark membaca dari HDFS, menggunakan DataFrame API + Spark SQL
Hasil disimpan ke HDFS /data/pangan/hasil/ dan dashboard/data/spark_results.json

In [1]:
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, LongType

## INISIALISASI SPARK SESSION


In [2]:
# Konfigurasi Spark dengan koneksi ke HDFS
spark = SparkSession.builder \
    .appName("HargaPangan-Analysis-Kelompok6") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 01:12:58 WARN Utils: Your hostname, LAPTOP-SPHS7VB5, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 01:12:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 01:12:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.setLogLevel("WARN")


## MEMBACA DATA DARI HDFS


In [4]:
HDFS_API_PATH = "hdfs://localhost:9000/data/pangan/api/"
HDFS_RSS_PATH = "hdfs://localhost:9000/data/pangan/rss/"
HDFS_HASIL_PATH = "hdfs://localhost:9000/data/pangan/hasil/"

In [9]:
# Membaca semua file JSON dari folder HDFS sekaligus
df_api = spark.read.option("multiLine", True).json(HDFS_API_PATH)
print(f"  → {df_api.count()} record API dimuat")
df_api.printSchema()
df_api.show(5, truncate=False)

  → 4 record API dimuat
root
 |-- harga: long (nullable = true)
 |-- jam: string (nullable = true)
 |-- komoditas: string (nullable = true)
 |-- label: string (nullable = true)
 |-- tanggal: string (nullable = true)

+-----+--------+---------+-------------+----------+
|harga|jam     |komoditas|label        |tanggal   |
+-----+--------+---------+-------------+----------+
|12000|10:00:00|beras    |Beras Premium|2026-04-28|
|13000|10:00:00|jagung   |Jagung Piper |2026-04-28|
|10500|11:00:00|beras    |Beras Premium|2026-04-28|
|11000|12:00:00|beras    |Beras Premium|2026-04-28|
+-----+--------+---------+-------------+----------+



In [10]:
df_rss = spark.read.option("multiLine", True).json(HDFS_RSS_PATH)
print(f"  → {df_rss.count()} record RSS dimuat")
df_rss.printSchema()
df_rss.show(5, truncate=False)

  → 3 record RSS dimuat
root
 |-- link: string (nullable = true)
 |-- title: string (nullable = true)

+------------+-------------------------------------+
|link        |title                                |
+------------+-------------------------------------+
|http://test |Harga beras naik tajam premium medium|
|http://test2|Petani jagung panen raya gila        |
|http://test3|Cabai merah memanas di pasar         |
+------------+-------------------------------------+



## PERSIAPAN DATA


In [11]:
# Pastikan kolom harga bertipe numerik
df_api = df_api.withColumn("harga", F.col("harga").cast(DoubleType()))

In [12]:
# Buat kolom datetime dari tanggal + jam (jika ada)
if "jam" in df_api.columns and "tanggal" in df_api.columns:
    df_api = df_api.withColumn(
        "datetime",
        F.to_timestamp(F.concat_ws(" ", F.col("tanggal"), F.col("jam")), "yyyy-MM-dd HH:mm:ss")
    )
    df_api = df_api.withColumn("hari", F.date_format(F.col("datetime"), "yyyy-MM-dd"))
    df_api = df_api.withColumn("jam_ke", F.hour(F.col("datetime")))
elif "tanggal" in df_api.columns:
    df_api = df_api.withColumn("hari", F.col("tanggal"))
    df_api = df_api.withColumn("jam_ke", F.lit(0))

In [13]:
# Register sebagai temp view untuk Spark SQL
df_api.createOrReplaceTempView("harga_pangan")
df_rss.createOrReplaceTempView("berita_pangan")

In [14]:
print("  → Temp view 'harga_pangan' dan 'berita_pangan' terdaftar")
print(f"  → Kolom API: {df_api.columns}")
print(f"  → Kolom RSS: {df_rss.columns}")

  → Temp view 'harga_pangan' dan 'berita_pangan' terdaftar
  → Kolom API: ['harga', 'jam', 'komoditas', 'label', 'tanggal', 'datetime', 'hari', 'jam_ke']
  → Kolom RSS: ['link', 'title']


In [15]:
# Daftar komoditas yang dipantau
KOMODITAS = ["beras", "jagung", "kedelai", "gula", "minyak_goreng", "cabai", "bawang_merah", "telur"]

## ANALISIS 1: VOLATILITAS HARGA PER KOMODITAS
(max_price - min_price) / avg_price * 100
Menggunakan DataFrame API


In [16]:
# Hitung indeks volatilitas relatif per komoditas
df_volatilitas = df_api.groupBy("komoditas", "label") \
    .agg(
        F.max("harga").alias("harga_max"),
        F.min("harga").alias("harga_min"),
        F.avg("harga").alias("harga_avg"),
        F.stddev("harga").alias("harga_stddev"),
        F.count("harga").alias("jumlah_data")
    ) \
    .withColumn(
        "indeks_volatilitas",
        F.round((F.col("harga_max") - F.col("harga_min")) / F.col("harga_avg") * 100, 2)
    ) \
    .withColumn("harga_max", F.round(F.col("harga_max"), 2)) \
    .withColumn("harga_min", F.round(F.col("harga_min"), 2)) \
    .withColumn("harga_avg", F.round(F.col("harga_avg"), 2)) \
    .withColumn("harga_stddev", F.round(F.col("harga_stddev"), 2)) \
    .orderBy(F.col("indeks_volatilitas").desc())

In [17]:
print("\n📊 Ranking Volatilitas Komoditas (tertinggi → terendah):")
df_volatilitas.show(truncate=False)


📊 Ranking Volatilitas Komoditas (tertinggi → terendah):
+---------+-------------+---------+---------+---------+------------+-----------+------------------+
|komoditas|label        |harga_max|harga_min|harga_avg|harga_stddev|jumlah_data|indeks_volatilitas|
+---------+-------------+---------+---------+---------+------------+-----------+------------------+
|beras    |Beras Premium|12000.0  |10500.0  |11166.67 |763.76      |3          |13.43             |
|jagung   |Jagung Piper |13000.0  |13000.0  |13000.0  |NULL        |1          |0.0               |
+---------+-------------+---------+---------+---------+------------+-----------+------------------+



In [18]:
# --- Narasi Interpretasi Bisnis ---
print("Interpretasi ANALISIS 1:")
vol_rows = df_volatilitas.collect()
if vol_rows:
    top = vol_rows[0]
    bottom = vol_rows[-1]
    print(f"""
Berdasarkan analisis volatilitas dari {sum(r['jumlah_data'] for r in vol_rows)} total data harga:
• Komoditas PALING VOLATIL: {top['label']} dengan indeks {top['indeks_volatilitas']}%
  Rentang harga: Rp {top['harga_min']:,.0f} - Rp {top['harga_max']:,.0f} (rata-rata Rp {top['harga_avg']:,.0f})
  Ini menunjukkan fluktuasi harga yang signifikan dan memerlukan perhatian khusus
  dari Bulog untuk stabilisasi pasokan.
• Komoditas PALING STABIL: {bottom['label']} dengan indeks {bottom['indeks_volatilitas']}%
  Rentang harga: Rp {bottom['harga_min']:,.0f} - Rp {bottom['harga_max']:,.0f}
  Stabilitas ini mengindikasikan rantai pasok yang baik dan intervensi yang efektif.
• Rekomendasi: Komoditas dengan indeks volatilitas > 10% sebaiknya mendapat
  prioritas intervensi harga dan buffer stok nasional.
""")

Interpretasi ANALISIS 1:

Berdasarkan analisis volatilitas dari 4 total data harga:
• Komoditas PALING VOLATIL: Beras Premium dengan indeks 13.43%
  Rentang harga: Rp 10,500 - Rp 12,000 (rata-rata Rp 11,167)
  Ini menunjukkan fluktuasi harga yang signifikan dan memerlukan perhatian khusus
  dari Bulog untuk stabilisasi pasokan.
• Komoditas PALING STABIL: Jagung Piper dengan indeks 0.0%
  Rentang harga: Rp 13,000 - Rp 13,000
  Stabilitas ini mengindikasikan rantai pasok yang baik dan intervensi yang efektif.
• Rekomendasi: Komoditas dengan indeks volatilitas > 10% sebaiknya mendapat
  prioritas intervensi harga dan buffer stok nasional.



## ANALISIS 2: RATA-RATA HARGA PER KOMODITAS PER PERIODE
Tren harga dari waktu ke waktu (groupBy komoditas + hari)
Menggunakan Spark SQL


In [19]:
# Query Spark SQL untuk tren harga per hari
query_tren = """
SELECT
    komoditas,
    label,
    hari,
    ROUND(AVG(harga), 2)   AS harga_rata2,
    ROUND(MIN(harga), 2)   AS harga_min,
    ROUND(MAX(harga), 2)   AS harga_max,
    COUNT(*)               AS jumlah_observasi
FROM harga_pangan
WHERE komoditas IS NOT NULL AND harga IS NOT NULL
GROUP BY komoditas, label, hari
ORDER BY komoditas, hari
"""
df_tren = spark.sql(query_tren)
print("\n📊 Rata-rata Harga per Komoditas per Hari:")
df_tren.show(50, truncate=False)


📊 Rata-rata Harga per Komoditas per Hari:
+---------+-------------+----------+-----------+---------+---------+----------------+
|komoditas|label        |hari      |harga_rata2|harga_min|harga_max|jumlah_observasi|
+---------+-------------+----------+-----------+---------+---------+----------------+
|beras    |Beras Premium|2026-04-28|11166.67   |10500.0  |12000.0  |3               |
|jagung   |Jagung Piper |2026-04-28|13000.0    |13000.0  |13000.0  |1               |
+---------+-------------+----------+-----------+---------+---------+----------------+



In [20]:
# Query untuk perubahan harga antar periode (jika multi-hari)
query_perubahan = """
SELECT
    komoditas,
    label,
    ROUND(AVG(harga), 2) AS harga_rata2_keseluruhan,
    ROUND(MIN(harga), 2) AS harga_terendah_sepanjang,
    ROUND(MAX(harga), 2) AS harga_tertinggi_sepanjang,
    COUNT(DISTINCT hari) AS jumlah_hari,
    COUNT(*) AS total_observasi
FROM harga_pangan
WHERE komoditas IS NOT NULL
GROUP BY komoditas, label
ORDER BY harga_rata2_keseluruhan DESC
"""
df_ringkasan_tren = spark.sql(query_perubahan)
print("📊 Ringkasan Tren Harga Keseluruhan:")
df_ringkasan_tren.show(truncate=False)

📊 Ringkasan Tren Harga Keseluruhan:
+---------+-------------+-----------------------+------------------------+-------------------------+-----------+---------------+
|komoditas|label        |harga_rata2_keseluruhan|harga_terendah_sepanjang|harga_tertinggi_sepanjang|jumlah_hari|total_observasi|
+---------+-------------+-----------------------+------------------------+-------------------------+-----------+---------------+
|jagung   |Jagung Piper |13000.0                |13000.0                 |13000.0                  |1          |1              |
|beras    |Beras Premium|11166.67               |10500.0                 |12000.0                  |1          |3              |
+---------+-------------+-----------------------+------------------------+-------------------------+-----------+---------------+



In [21]:
# --- Narasi Interpretasi Bisnis ---
print("Interpretasi ANALISIS 2:")
tren_rows = df_ringkasan_tren.collect()
if tren_rows:
    termahal = tren_rows[0]
    termurah = tren_rows[-1]
    print(f"""
Analisis tren harga berdasarkan data dari {termahal['jumlah_hari']} hari pengamatan:
• Komoditas TERMAHAL rata-rata: {termahal['label']}
  Harga rata-rata: Rp {termahal['harga_rata2_keseluruhan']:,.0f}
  dengan {termahal['total_observasi']} titik data terekam.
• Komoditas TERMURAH rata-rata: {termurah['label']}
  Harga rata-rata: Rp {termurah['harga_rata2_keseluruhan']:,.0f}
• Tren ini membantu Bulog dalam merencanakan anggaran pengadaan dan
  menentukan timing intervensi pasar yang optimal. Data per-periode
  memungkinkan deteksi pola musiman dan anomali harga dini.
""")

Interpretasi ANALISIS 2:

Analisis tren harga berdasarkan data dari 1 hari pengamatan:
• Komoditas TERMAHAL rata-rata: Jagung Piper
  Harga rata-rata: Rp 13,000
  dengan 1 titik data terekam.
• Komoditas TERMURAH rata-rata: Beras Premium
  Harga rata-rata: Rp 11,167
• Tren ini membantu Bulog dalam merencanakan anggaran pengadaan dan
  menentukan timing intervensi pasar yang optimal. Data per-periode
  memungkinkan deteksi pola musiman dan anomali harga dini.



## ANALISIS 3: SEBUTAN KOMODITAS DI BERITA
Korelasi frekuensi berita dengan perubahan harga
Menggunakan kombinasi DataFrame API + Spark SQL


Hitung kemunculan nama komoditas di judul berita RSS
Menggunakan DataFrame API untuk string matching

In [22]:
komoditas_patterns = {
    "beras": "beras",
    "jagung": "jagung",
    "kedelai": "kedelai",
    "gula": "gula",
    "minyak_goreng": "minyak",
    "cabai": "cabai",
    "bawang_merah": "bawang",
    "telur": "telur"
}

In [23]:
# Buat kolom flag untuk setiap komoditas yang disebut di title
title_col = "title" if "title" in df_rss.columns else df_rss.columns[0]

In [24]:
df_berita_tagged = df_rss
for kom, keyword in komoditas_patterns.items():
    df_berita_tagged = df_berita_tagged.withColumn(
        f"sebut_{kom}",
        F.when(F.lower(F.col(title_col)).contains(keyword), 1).otherwise(0)
    )

In [25]:
# Hitung frekuensi sebutan per komoditas
sebutan_data = []
for kom in KOMODITAS:
    count_val = df_berita_tagged.agg(F.sum(f"sebut_{kom}")).collect()[0][0] or 0
    sebutan_data.append((kom, komoditas_patterns[kom], int(count_val)))

In [26]:
df_sebutan = spark.createDataFrame(sebutan_data, ["komoditas", "keyword", "jumlah_sebutan"])
df_sebutan.createOrReplaceTempView("sebutan_berita")

In [27]:
# Gabungkan dengan data volatilitas menggunakan Spark SQL
df_volatilitas.createOrReplaceTempView("volatilitas")

In [28]:
query_korelasi = """
SELECT
    v.komoditas,
    v.label,
    v.indeks_volatilitas,
    v.harga_avg,
    s.jumlah_sebutan,
    CASE
        WHEN s.jumlah_sebutan > 5 AND v.indeks_volatilitas > 5
            THEN 'TINGGI — berita banyak & harga bergejolak'
        WHEN s.jumlah_sebutan > 5 AND v.indeks_volatilitas <= 5
            THEN 'MODERAT — banyak diberitakan tapi harga stabil'
        WHEN s.jumlah_sebutan <= 5 AND v.indeks_volatilitas > 5
            THEN 'WASPADA — harga bergejolak tapi minim liputan'
        ELSE 'RENDAH — stabil dan minim berita'
    END AS status_korelasi
FROM volatilitas v
JOIN sebutan_berita s ON v.komoditas = s.komoditas
ORDER BY s.jumlah_sebutan DESC, v.indeks_volatilitas DESC
"""
df_korelasi = spark.sql(query_korelasi)
print("\n📊 Korelasi Sebutan Berita vs Volatilitas Harga:")
df_korelasi.show(truncate=False)


📊 Korelasi Sebutan Berita vs Volatilitas Harga:


+---------+-------------+------------------+---------+--------------+---------------------------------------------+
|komoditas|label        |indeks_volatilitas|harga_avg|jumlah_sebutan|status_korelasi                              |
+---------+-------------+------------------+---------+--------------+---------------------------------------------+
|beras    |Beras Premium|13.43             |11166.67 |1             |WASPADA — harga bergejolak tapi minim liputan|
|jagung   |Jagung Piper |0.0               |13000.0  |1             |RENDAH — stabil dan minim berita             |
+---------+-------------+------------------+---------+--------------+---------------------------------------------+



In [29]:
# Tampilkan total berita
total_berita = df_rss.count()
print(f"Total artikel berita RSS yang dianalisis: {total_berita}")

Total artikel berita RSS yang dianalisis: 3


In [30]:
# --- Narasi Interpretasi Bisnis ---
print("\n📝 NARASI INTERPRETASI ANALISIS 3:")
kor_rows = df_korelasi.collect()
if kor_rows:
    paling_disebut = kor_rows[0]
    waspada = [r for r in kor_rows if "WASPADA" in r["status_korelasi"]]
    print(f"""
Analisis korelasi berita terhadap {total_berita} artikel RSS:
• Komoditas PALING BANYAK diberitakan: {paling_disebut['label']}
  Disebut {paling_disebut['jumlah_sebutan']}x dalam judul berita
  dengan indeks volatilitas {paling_disebut['indeks_volatilitas']}%.
  Status: {paling_disebut['status_korelasi']}
""")
    if waspada:
        print("⚠️  PERINGATAN — Komoditas berikut bergejolak TANPA liputan media:")
        for w in waspada:
            print(f"   • {w['label']}: volatilitas {w['indeks_volatilitas']}%, hanya {w['jumlah_sebutan']}x disebut")
        print("   → Ini bisa menjadi blind-spot bagi pengambil kebijakan.\n")
    print("""• Insight: Korelasi positif antara frekuensi pemberitaan dan volatilitas
  menunjukkan media cukup responsif terhadap gejolak harga. Namun komoditas
  dengan status 'WASPADA' perlu mendapat perhatian ekstra karena fluktuasi
  harganya tidak terdeteksi oleh media publik.
""")


📝 NARASI INTERPRETASI ANALISIS 3:

Analisis korelasi berita terhadap 3 artikel RSS:
• Komoditas PALING BANYAK diberitakan: Beras Premium
  Disebut 1x dalam judul berita
  dengan indeks volatilitas 13.43%.
  Status: WASPADA — harga bergejolak tapi minim liputan

⚠️  PERINGATAN — Komoditas berikut bergejolak TANPA liputan media:
   • Beras Premium: volatilitas 13.43%, hanya 1x disebut
   → Ini bisa menjadi blind-spot bagi pengambil kebijakan.

• Insight: Korelasi positif antara frekuensi pemberitaan dan volatilitas
  menunjukkan media cukup responsif terhadap gejolak harga. Namun komoditas
  dengan status 'WASPADA' perlu mendapat perhatian ekstra karena fluktuasi
  harganya tidak terdeteksi oleh media publik.



## SIMPAN HASIL KE HDFS


In [31]:
# Simpan ke HDFS dalam format JSON
df_volatilitas.coalesce(1).write.mode("overwrite").json(HDFS_HASIL_PATH + "volatilitas")
print("  → Volatilitas tersimpan ke HDFS: /data/pangan/hasil/volatilitas/")

  → Volatilitas tersimpan ke HDFS: /data/pangan/hasil/volatilitas/


In [32]:
df_tren.coalesce(1).write.mode("overwrite").json(HDFS_HASIL_PATH + "tren_harga")
print("  → Tren harga tersimpan ke HDFS: /data/pangan/hasil/tren_harga/")

  → Tren harga tersimpan ke HDFS: /data/pangan/hasil/tren_harga/


In [33]:
df_korelasi.coalesce(1).write.mode("overwrite").json(HDFS_HASIL_PATH + "korelasi_berita")
print("  → Korelasi berita tersimpan ke HDFS: /data/pangan/hasil/korelasi_berita/")

  → Korelasi berita tersimpan ke HDFS: /data/pangan/hasil/korelasi_berita/


## SIMPAN RINGKASAN UNTUK DASHBOARD


In [37]:
# Konversi ke Pandas lalu simpan sebagai JSON lokal
spark_results = {
    "metadata": {
        "generated_by": "Theodorus Aaron Ugraha (Anggota 4)",
        "pipeline": "HargaPangan-Kelompok6",
        "total_data_api": df_api.count(),
        "total_data_rss": df_rss.count(),
    },
    "analisis_1_volatilitas": df_volatilitas.toPandas().to_dict(orient="records"),
    "analisis_2_tren_harga": df_ringkasan_tren.toPandas().to_dict(orient="records"),
    "analisis_3_korelasi_berita": df_korelasi.toPandas().to_dict(orient="records"),
}

In [39]:
dashboard_data_dir = os.path.join(os.path.abspath(os.getcwd()), "..", "dashboard", "data")
os.makedirs(dashboard_data_dir, exist_ok=True)

In [40]:
output_path = os.path.join(dashboard_data_dir, "spark_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2, default=str)

In [41]:
print(f"  → Tersimpan: {output_path}")

  → Tersimpan: /home/aaron31/BigDat/ETS/BigData-ETS/spark/../dashboard/data/spark_results.json


In [42]:
print(f"  • HDFS: {HDFS_HASIL_PATH}volatilitas/")
print(f"  • HDFS: {HDFS_HASIL_PATH}tren_harga/")
print(f"  • HDFS: {HDFS_HASIL_PATH}korelasi_berita/")
print(f"  • Lokal: {output_path}")


  • HDFS: hdfs://localhost:9000/data/pangan/hasil/volatilitas/
  • HDFS: hdfs://localhost:9000/data/pangan/hasil/tren_harga/
  • HDFS: hdfs://localhost:9000/data/pangan/hasil/korelasi_berita/
  • Lokal: /home/aaron31/BigDat/ETS/BigData-ETS/spark/../dashboard/data/spark_results.json


In [43]:
spark.stop()
print("Spark session ditutup.")

Spark session ditutup.
